# Uncertainty Quantification for LLMs: A Comprehensive Tutorial

Welcome to this interactive tutorial on estimating uncertainty in Large Language Models (LLMs) applied to the clinical domain. 

This notebook provides a hands-on approach to Uncertainty Quantification (UQ) by leveraging two state-of-the-art Python libraries: **`lm-polygraph`** and **`uqlm`**.

---

### Global Configuration
Before diving into the code, we need to set up our experimental environment. 
Use the interactive dashboard below to define your global configuration:
1. **Select an LLM:** Choose a suggested clinical model from the list or type any valid Hugging Face model ID (e.g., `Qwen/Qwen2.5-0.5B` for a quick, lightweight test).
2. **Select Granularity:** Choose whether to analyze uncertainty at the Token, Sequence, or Claim level.
3. **Confirm:** Click the button to save these settings globally for the rest of the tutorial.

In [ ]:
import lm_polygraph
import uqlm
import torch
import random
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ==========================================
# CELL 1: SETUP AND CONFIGURATION 
# Run this cell to load libraries and initialize the UI.
# ==========================================


# --- 1. Scientific Reproducibility ---
RANDOM_SEED = 42
def set_global_seed(seed=RANDOM_SEED):
    """Locks the random seed for predictable, reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_global_seed()

# --- 2. Global Configuration Data ---
SUPPORTED_MODELS = [
    "google/medgemma-2b", 
    "epfl-llm/meditron-7b", 
    "llava-hf/llava-1.5-7b-hf"
]
GRANULARITIES = ["Token", "Sequence", "Claim"]

# --- 3. Interactive User Interface (UI) ---
model_input = widgets.Combobox(
    placeholder='Select or type a Hugging Face Model ID',
    options=SUPPORTED_MODELS,
    value=SUPPORTED_MODELS[0],
    description='Select LLM:',
    ensure_option=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

granularity_dropdown = widgets.Dropdown(
    options=GRANULARITIES,
    value=GRANULARITIES[0],
    description='Granularity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

confirm_button = widgets.Button(
    description='Confirm & Save',
    button_style='primary',
    icon='check',
    layout=widgets.Layout(width='400px')
)

output_console = widgets.Output()

# Global variables to store user selections
SELECTED_MODEL = None
SELECTED_GRANULARITY = None

def on_confirm_clicked(b):
    """Handles the configuration saving process when the button is clicked."""
    global SELECTED_MODEL, SELECTED_GRANULARITY
    with output_console:
        clear_output()
        SELECTED_MODEL = model_input.value.strip()
        SELECTED_GRANULARITY = granularity_dropdown.value
        
        if not SELECTED_MODEL:
            print("❌ Error: Please enter a valid model ID.")
            return
            
        print("✅ Configuration Saved Successfully!")
        print(f"Active Model: {SELECTED_MODEL}")
        print(f"Target Granularity: {SELECTED_GRANULARITY}-level")
        print("-" * 50)
        print("Proceed to the next cell to allocate the model in memory.")

confirm_button.on_click(on_confirm_clicked)

# Assemble the UI components
dashboard = widgets.VBox([
    widgets.HTML("<h3>Step 1: Global Configuration</h3><p>Select the underlying model and the target granularity level.</p>"),
    model_input, 
    granularity_dropdown, 
    confirm_button, 
    output_console
])

# Render the dashboard
display(dashboard)

### Hugging Face Authentication (For Gated Models)

Many state-of-the-art clinical models (such as Google's `medgemma-2b` or Meta's `Llama-3`) are **gated repositories**. This means you cannot download their weights anonymously; you must explicitly agree to their usage terms.

**Prerequisites to load gated models:**
1. Log in to your [Hugging Face](https://huggingface.co/) account.
2. Navigate to the specific model's page (e.g., `google/medgemma-2b`) and click **"Acknowledge license"**.
3. Go to your profile **Settings > Access Tokens** and generate a new token (Read permission).
4. Run the cell below and securely paste your token.

> ⚠️ **Security Note:** We use the `getpass` library to securely input your token. Never hardcode your personal tokens directly into the notebook cells to prevent accidental leaks on GitHub.

In [ ]:
import getpass
from huggingface_hub import login

# Securely prompt the user for their Hugging Face Token
print("Please paste your Hugging Face Access Token below:")
HF_TOKEN = getpass.getpass("Token: ")

# Perform the login to Hugging Face Hub
login(token=HF_TOKEN)

print("✅ Authentication Completed! You are now authorized to download gated models.")

Please paste your Hugging Face Access Token below:


---

### Hardware Allocation & Model Loading



By executing the explicit code cell below, you can:
1. Monitor the real-time download progress from Hugging Face.
2. Learn the standard `transformers` library syntax for model initialization.
3. See how we manage hardware limits by automatically mapping weights across available memory (`device_map="auto"`).

**Action:** Run the cell below to physically allocate the selected model into your system's memory.

In [ ]:
# ==========================================
# AUTHENTICATION & MODEL LOADING
# Run this cell to authenticate and allocate the model to GPU memory.
# ==========================================

# Retrieve the global variable set by the dashboard in Cell 1
if SELECTED_MODEL is None:
    raise ValueError("❌ No model selected. Please go back to Step 1, select a model, and click 'Confirm & Save'.")

print(f"🔄 Downloading and initializing model: '{SELECTED_MODEL}'")
print("⏳ Allocating weights to GPU memory (this may take a few minutes)...")

# --- 3. Model & Tokenizer Initialization ---
# Best Practice: We use float16 precision and automatic device mapping 
# to prevent Out-Of-Memory (OOM) errors on standard GPUs.

try:
    tokenizer = AutoTokenizer.from_pretrained(SELECTED_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(
        SELECTED_MODEL,
        device_map="auto"            # Automatically maps layers to GPU/CPU
    )
    print("✅ Model successfully loaded and ready for Uncertainty Quantification!")
    
except Exception as e:
    print(f"❌ A critical error occurred while loading the model:\n{e}")

🔄 Downloading and initializing model: 'Qwen/Qwen2.5-0.5B'
⏳ Allocating weights to GPU memory (this may take a few minutes)...
✅ Model successfully loaded and ready for Uncertainty Quantification!
